# EmbedCatalog — Full Artifact Re-run (Colab, GPU)

Regenerates the complete artifact so the paper reproduces end to end. Run the cells top to bottom.

**Why this notebook exists:** the committed `results/` came from a partial run (FAISS not installed; float16/OPQ not yet in the sweep; the index/RQ4 sweep failed). This re-run fixes all of that and regenerates every table and figure from one canonical run.

**GPU note:** the GPU accelerates the embedding/scoring path (the dominant cost). The ANN backends use **faiss-cpu** and run on CPU — expected, and matches the paper's FAISS-family scope. Prefer **A100 / High-RAM** (Colab Pro) for TREC-COVID, FiQA, and the 9-pair seed-variance grid.


## 0 — Confirm GPU runtime
Runtime ▸ Change runtime type ▸ **GPU** (A100 preferred).

In [ ]:
!nvidia-smi

## 1 — Clone + install (the fix: the `index` extra installs faiss-cpu, which was missing before)

In [ ]:
!git clone https://github.com/jorge-martinez-gil/embeddings.git
%cd embeddings
!pip -q install -e ".[paper,index]"
import faiss, torch
print("faiss", faiss.__version__, "| torch", torch.__version__, "| cuda", torch.cuda.is_available())


## 2 — (Recommended) persist results + cache on Drive so a disconnect never loses work
The embedding pass is the expensive step; keeping it on Drive makes resumes cheap.

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
import os
PERSIST='/content/drive/MyDrive/embedopt_run'
os.makedirs(f'{PERSIST}/results', exist_ok=True)
# send results to Drive (the runner checkpoints per spec/pair, so re-running resumes)
!rm -rf results && ln -s {PERSIST}/results results
print("results ->", os.path.realpath('results'))


## 3 — Download BEIR datasets to /content (the `beir-local:` loader takes a path)

In [ ]:
!pip -q install beir
import os
from beir import util
base='/content/datasets'; os.makedirs(base, exist_ok=True)
URL='https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/{}.zip'
for d in ['scifact','nfcorpus','arguana','fiqa','trec-covid']:
    util.download_and_unzip(URL.format(d), base)
!ls -1 /content/datasets


## 4 — Main sweep: all backbones × main datasets, all six backends, paper-grade resamples
*Resumable:* checkpoints per spec and per (backbone, dataset). If the session drops, re-run this cell — completed pairs are skipped (omit `--force` to resume; keep it only for a clean rebuild). For very long jobs, split into one backbone per cell.

In [ ]:
!python scripts/run_paper_experiments.py \
  --backbones e5-base bge-base mxbai-large \
  --datasets beir-local:/content/datasets/scifact beir-local:/content/datasets/nfcorpus beir-local:/content/datasets/arguana \
  --index-backends exact-numpy faiss-flat faiss-ivf faiss-ivfpq faiss-hnsw faiss-opq \
  --bootstrap-resamples 5000 --significance-resamples 5000 \
  --profile-repeats 20 --score-device cuda --force \
  --output-dir results


## 5 — Robustness datasets (E5-base only)

In [ ]:
!python scripts/run_paper_experiments.py \
  --backbones e5-base \
  --datasets beir-local:/content/datasets/fiqa beir-local:/content/datasets/trec-covid \
  --index-backends exact-numpy faiss-flat faiss-ivf faiss-ivfpq faiss-hnsw faiss-opq \
  --bootstrap-resamples 5000 --significance-resamples 5000 \
  --profile-repeats 20 --score-device cuda --force \
  --output-dir results


## 6 — Seed-variance grid (PQ + OPQ) for all nine main pairs
The paper generalizes the seed-variance claim across pairs, so run all nine (the committed run had only e5/scifact).

In [ ]:
%%bash
for BB in e5-base bge-base mxbai-large; do
  for DS in scifact nfcorpus arguana; do
    python scripts/run_seed_variance.py --backbone $BB \
      --dataset beir-local:/content/datasets/$DS --seeds 0 1 2 3 4 --output-dir results
  done
done


## 7 — Storage-mode comparison (one corpus → `fig:storage-modes`)
The storage-mode bar chart is defined on a single embedded corpus. Per-pair float16 equivalence is covered separately by the main sweep + TOST (Cell 9).

In [ ]:
!python scripts/compare_storage_modes.py \
  --backbone e5-base --dataset beir-local:/content/datasets/scifact --seed 0 \
  --output results/storage_modes_comparison.csv
# Optional — emit a per-pair storage-mode CSV for extra float16 backing:
# %%bash
# for DS in scifact nfcorpus arguana; do
#   python scripts/compare_storage_modes.py --backbone e5-base \
#     --dataset beir-local:/content/datasets/$DS --seed 0 \
#     --output results/storage_modes__e5-base__$DS.csv
# done


## 8 — Coverage sanity check (must pass before regenerating tables)
Every `candidates.csv` should now contain float16 + OPQ, and every `index.csv` should have `ok` rows for all six backends.

In [ ]:
import csv, glob, os
from collections import Counter
print("== candidate sweeps ==")
for f in sorted(glob.glob("results/*__candidates.csv")):
    s=[r['spec'] for r in csv.DictReader(open(f))]
    print(os.path.basename(f), len(s),
          "float16" if any('float16' in x for x in s) else "NO-float16",
          "opq" if any('opq' in x.lower() for x in s) else "NO-opq")
print("== index backends (ok rows) ==")
for f in sorted(glob.glob("results/*__index.csv")):
    ok=Counter(r['index_backend'] for r in csv.DictReader(open(f)) if r['index_status']=='ok')
    print(os.path.basename(f), dict(ok))


## 9 — Regenerate the entire empirical section (tables + TOST + figures)
One command per artifact type. `regen_tables.py` and `tost_equivalence.py` auto-pick-up float16/OPQ rows now that they exist.

In [ ]:
!python scripts/regen_tables.py --emit-tex results/paper_tables.tex
!python scripts/tost_equivalence.py --results results --spec scalar_int8 --eps 0.01 --eps2 0.005 --out results/tost_scalar_int8.csv
!python scripts/tost_equivalence.py --results results --spec float16     --eps 0.01 --eps2 0.005 --out results/tost_float16.csv
!python scripts/plot_paper_figures.py --results-dir results --output-dir figures
print("\nIn the manuscript:  \\input{results/paper_tables.tex}   (or paste each block)")


## 10 — Download the regenerated artifact
(Skip if you mounted Drive in Cell 2 — it's already saved there.)

In [ ]:
!zip -qr artifact_regen.zip results figures
from google.colab import files; files.download('artifact_regen.zip')


## Watch-outs
- **Session timeouts.** Free Colab idles out (~12 h cap). Use Drive (Cell 2) and run heavy pairs in separate cells so a drop never loses a completed pair.
- **Re-embedding cost.** A fresh VM loses the embedding cache unless it's on Drive — persisting it makes re-runs cheap.
- **Don't install faiss-gpu.** The `index` extra's faiss-cpu is what the backends use; faiss-gpu can clash with Colab's CUDA/torch build.
- **HV reference point.** After regen, confirm the figure/runner use the same hypervolume reference point the paper text assumes (the committed HV differed ~2.7×).
- **Final gate.** Follow `RECONCILIATION_AND_RERUN.md` §5 verification checklist before submitting.
